# YOLO Custom Model Training Manual
This notebook is a step-by-step guide to training, validating, testing, and exporting a custom YOLO object detection model.

In this guide, we will use your custom dataset (`overall-ptt-object-detection`) and the transfer learning technique with the pre-trained `yolo26n.pt` weights.

## 1. Setup and Initialization
First, we need to import the Ultralytics library and initialize our model. We will load the pre-trained `yolo26n.pt` model which already has general vision capabilities.

In [ ]:
from ultralytics import YOLO

# Load a pre-trained model (recommended for training)
model = YOLO('yolo26n.pt')

## 2. Train the Model
Now we train the model on our custom data. We pass the path to our `data.yaml` file, and specify the number of `epochs` (how many times the model sees the entire dataset) and the `imgsz` (image resolution).

> **Tip:** For small datasets, 50-100 epochs is usually a good starting point.

In [ ]:
# Define the path to your dataset configuration file
data_path = '/home/luke/ai_training/ultralytics/datasets/coco8/overall-ptt-object-detection.v11i.yolov11/data.yaml'

# Train the model
results = model.train(
    data=data_path,
    epochs=50,       # Number of training epochs
    imgsz=640,       # Image size for training
    batch=16,        # Number of images per batch
    name='ptt_custom_model' # Name of the folder where results will be saved
)

## 3. Validate the Model
After training, it's important to validate the model to see how well it performs on images it hasn't trained on (the validation set). 

> **Important:** Always pass the `data` argument when validating. If you omit it, YOLO will default to the standard 80-class COCO dataset, which will cause an `IndexError` because your custom model has a different number of classes!

In [ ]:
# Load the best weights from our training run
# (Replace 'runs/detect/train-3/weights/best.pt' with your actual path if different)
best_model = YOLO('runs/detect/train-3/weights/best.pt')

# Define the path to your dataset configuration file
data_path = '/home/luke/ai_training/ultralytics/datasets/coco8/overall-ptt-object-detection.v11i.yolov11/data.yaml'

# Validate the model (Always pass the data argument to avoid class mismatch errors!)
metrics = best_model.val(data=data_path)

print(f"Mean Average Precision (mAP50-95): {metrics.box.map:.4f}")

## 4. Predict (Inference)
Now for the fun part! Let's use our trained model to make predictions on new images. You can pass a path to a specific image, a folder of images, or even a video.

In [ ]:
# Run prediction on an image (replace with a real image path from your test set)
test_image = '/home/luke/ai_training/ultralytics/datasets/coco8/overall-ptt-object-detection.v11i.yolov11/test/images/YOUR_IMAGE_HERE.jpg'

# Perform inference and save the result
prediction_results = best_model.predict(source=test_image, save=True, conf=0.5)

# The output will be saved in runs/detect/predict/

## 5. Export the Model
Once you are happy with the model, you will likely want to deploy it. You can export the `.pt` file to different formats like ONNX, TensorRT, or OpenVINO depending on your hardware.

In [ ]:
# Export the model to ONNX format
export_path = best_model.export(format='onnx')
print(f"Model exported to: {export_path}")